In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_region(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/region.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
region = load_region(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Precompute timestamp boundaries
start = pd.Timestamp('1995-01-01')
end = pd.Timestamp('1997-01-01')

# Identify relevant keys via vectorized filters
economy_parts = set(part.loc[part['P_TYPE'] == 'ECONOMY ANODIZED STEEL', 'P_PARTKEY'])
america_regions = set(region.loc[region['R_NAME'] == 'AMERICA', 'R_REGIONKEY'])
america_nations = set(nation.loc[nation['N_REGIONKEY'].isin(america_regions), 'N_NATIONKEY'])
america_customers = set(customer.loc[customer['C_NATIONKEY'].isin(america_nations), 'C_CUSTKEY'])

# Filter orders by date range and American customers, extract year
orders_america = (
    orders
    .loc[
        (orders['O_ORDERDATE'] >= start)
        & (orders['O_ORDERDATE'] < end)
        & (orders['O_CUSTKEY'].isin(america_customers)),
        ['O_ORDERKEY', 'O_ORDERDATE']
    ]
    .assign(O_YEAR=lambda df: df['O_ORDERDATE'].dt.year)
    [['O_ORDERKEY', 'O_YEAR']]
)

# Prepare supplier nation names lookup
nation_names = (
    nation[['N_NATIONKEY', 'N_NAME']]
    .rename(columns={'N_NAME': 'NATION'})
)

# Build a single dataframe of (VOLUME, O_YEAR, supplier NATION)
df = (
    lineitem
    # keep only lineitems for the chosen parts
    .loc[lineitem['L_PARTKEY'].isin(economy_parts),
         ['L_SUPPKEY', 'L_ORDERKEY', 'L_EXTENDEDPRICE', 'L_DISCOUNT']]
    # compute volume
    .assign(VOLUME=lambda df: df['L_EXTENDEDPRICE'] * (1 - df['L_DISCOUNT']))
    [['L_SUPPKEY', 'L_ORDERKEY', 'VOLUME']]
    # attach supplier nation key
    .merge(
        supplier[['S_SUPPKEY', 'S_NATIONKEY']],
        left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner'
    )[['L_ORDERKEY', 'VOLUME', 'S_NATIONKEY']]
    # attach order year (only American customers in date range)
    .merge(
        orders_america,
        left_on='L_ORDERKEY', right_on='O_ORDERKEY', how='inner'
    )[['VOLUME', 'S_NATIONKEY', 'O_YEAR']]
    # attach supplier nation name
    .merge(
        nation_names,
        left_on='S_NATIONKEY', right_on='N_NATIONKEY', how='inner'
    )[['VOLUME', 'O_YEAR', 'NATION']]
)

# Compute, per year, total volume and volume for BRAZIL, then market share
denominator = df.groupby('O_YEAR', sort=True)['VOLUME'].sum()
numerator = (
    df.loc[df['NATION'] == 'BRAZIL']
      .groupby('O_YEAR')['VOLUME']
      .sum()
)

mkt_share = (numerator / denominator).fillna(0)
mkt_share.index.name = 'O_YEAR'
mkt_share.name = 'MKT_SHARE'

total = mkt_share.reset_index().sort_values('O_YEAR')